#Instalação de Bibliotecas e imports#
Aqui será instalado todas as bibliotecas necessárias para a execução do código de Fine Tunning.

In [1]:
!pip install unsloth
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install transformers datasets
!pip install posthog==7.0.0
!pip install deepeval
!pip install google.generativeai
#!pip install chromadb
!pip install google.api_core



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.5/317.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.5/256.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 21.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    

In [2]:
# Importações iniciais
import json
import re
import html
import unicodedata
import os
import torch
import gc
import os
from google.colab import userdata
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments,AutoTokenizer


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


#Preparação dos datasets

##Acesso ao Google Drive

In [3]:
# Montagem do Google Drive para acesso aos arquivos
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


## Configurando os diretórios

*   Arquivo bruto de input - input_file_path
*   Arquivo gerado com a limpeza dos campos - input_file_path
*   Dataset final com os prompts para treinamento - final_dataset_path
*   Local onde será salvo o modelo após o treinamento - model_directory


In [4]:
# Definindo os caminhos dos arquivos
input_file_path = '/content/drive/MyDrive/FIAP/TechChallengeFineTunning/trn.json'
structured_prompts_path = '/content/drive/MyDrive/FIAP/TechChallengeFineTunning/structured_prompts.json'

# Caminho para o dataset formatado que será usado pelo trainer.
# Este arquivo será gerado pela função `formatting_prompts_func`.
final_dataset_path = "/content/drive/MyDrive/FIAP/TechChallengeFineTunning/final_dataset_for_trainer.json"

# Define o caminho no Google Drive onde o modelo será salvo.
# Certifique-se de que o caminho existe e que você tem permissão para escrever nele.
model_directory = "/content/drive/MyDrive/FIAP/TechChallengeFineTunning/trained_model"

#Criação dos Datasets de treinamento

In [5]:
# --- Formatação do Prompt no Estilo Alpaca ---

# Estrutura de prompt inspirada no modelo Alpaca. Ajuda o modelo a entender a tarefa.
alpaca_prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request. If you don't know the answer to a question, please don't share false information.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# O tokenizador precisa saber onde termina uma sequência para não gerar texto indefinidamente.
EOS_TOKEN = "<|end_of_text|>"

# Instrução utilizada como primeiro parametro do prompt estilo Alpaca
instruction = "Generate a detailed and informative description for the provided book title."

def clean_and_normalize_text(text: str) -> str:
    """
    Realiza uma limpeza completa no texto, removendo espaços excessivos,
    convertendo entidades HTML e normalizando caracteres Unicode.
    """
    # Substitui múltiplos espaços, tabulações e quebras de linha por um único espaço
    text = re.sub(r'\s+', ' ', text)
    # Remove espaços em branco no início e no final do texto
    text = text.strip()
    # Converte entidades HTML (ex: &amp;) para seus caracteres correspondentes (ex: &)
    text = html.unescape(text)
    # Normaliza caracteres Unicode para a forma mais simples (ex: remove acentos)
    # e converte para ASCII, ignorando caracteres que não podem ser representados.
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    # Codifica e decodifica para garantir consistência na representação de caracteres de escape
    text = text.encode('unicode_escape').decode('unicode_escape')
    return text

def create_instructional_prompts_from_json(json_input_path: str, json_output_path: str, num_samples: int = 100_000):
    """
    Lê o arquivo JSON bruto, limpa os campos 'title' e 'content',
    e formata os dados em prompts instrucionais para o fine-tuning.
    """
    # Lista para armazenar os dados processados
    processed_items = []

    # Abre o arquivo JSON de entrada para leitura
    with open(json_input_path, "r", encoding="utf-8") as file:
        # Lê todas as linhas do arquivo
        lines = file.readlines()

    # Itera sobre um subconjunto das linhas do arquivo para agilizar o processo
    for line in lines[:num_samples]:
        item = json.loads(line)  # Converte a linha JSON em um dicionário Python

        # Extrai e limpa o título e o conteúdo
        title = clean_and_normalize_text(item.get('title', ''))
        content = clean_and_normalize_text(item.get('content', ''))

        # Garante que tanto o título quanto o conteúdo existam e não sejam idênticos
        if title and content and title != content:
            # Cria o prompt com marcadores personalizados para separar claramente as seções
            item["instruction"] = instruction
            item["input"] = title
            item["output"] = content
            processed_items.append(item)

    # Salva a lista de prompts processados em um novo arquivo JSON
    with open(json_output_path, "w", encoding="utf-8") as output_file:
        json.dump(processed_items, output_file, ensure_ascii=False, indent=4)
    print(f"Prompts instrucionais salvos em: {json_output_path}")


# Executa a função que cria o dataset formatado (caso ele não exista) a partir do dataset inicial. Esse novo datasaet terá os campos instruction, input e output.
if not os.path.exists(structured_prompts_path):
  create_instructional_prompts_from_json(input_file_path, structured_prompts_path, 100_000)


# Carrega o dataset formatado
dataset_formatado = None
dataset_formatado = load_dataset("json", data_files=structured_prompts_path, split="train")

# Define um título de exemplo para ser utilizado nos testes ao longo do código
title_question_example = dataset_formatado[24]["title"]
print(f"Título/Pergunta:\n {title_question_example}")
print(f"Conteúdo:\n{dataset_formatado[24]["content"]}")

Generating train split: 0 examples [00:00, ? examples/s]

Título/Pergunta:
 Birds of Britain and Europe with North Africa and the Middle East: Over 3,000 Colour Illustrations (Collins Pocket Guide)
Conteúdo:
Brilliantly organised guide ... beautiful, apparently infallible, so comprehensive and above all easy to use. The Sunday Times  Superbly handy and comprehensive guide. The Observer


In [6]:
#Gerando dataset com o prompt final
def formatting_prompts_for_trainer(examples):
    """
    Formata o dataset para o formato de texto único exigido pelo SFTTrainer,
    usando o template Alpaca.
    """
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt_template.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Aplica a formatação ao dataset
dataset_com_prompt = None
dataset_com_prompt = dataset_formatado.map(formatting_prompts_for_trainer, batched = True,)
# Manter apenas a coluna 'text' no dataset formatado
dataset_com_prompt = dataset_com_prompt.select_columns(['text'])
torch.cuda.empty_cache()

Map:   0%|          | 0/75058 [00:00<?, ? examples/s]

In [7]:
print(dataset_com_prompt["text"])

Column(["Below is an instruction that describes a task, paired with an input that provides further context.\nWrite a response that appropriately completes the request. If you don't know the answer to a question, please don't share false information.\n\n### Instruction:\nGenerate a detailed and informative description for the provided book title.\n\n### Input:\nGirls Ballet Tutu Neon Pink\n\n### Response:\nHigh quality 3 layer ballet tutu. 12 inches in length<|end_of_text|>", "Below is an instruction that describes a task, paired with an input that provides further context.\nWrite a response that appropriately completes the request. If you don't know the answer to a question, please don't share false information.\n\n### Instruction:\nGenerate a detailed and informative description for the provided book title.\n\n### Input:\nMog's Kittens\n\n### Response:\nJudith Kerrs bestselling adventures of that endearing (and exasperating) cat Mog have entertained children for more than 30 years. No

#Configuração do Modelo e Treinamento (Fine Tunning)

In [10]:
# --- Parâmetros de Configuração do Modelo e Treinamento ---

# (Opcional) Modelo pré-quantizado da biblioteca Unsloth.
# Unsloth oferece variantes de modelos populares já otimizados para 4 bits.
# model_name = "unsloth/llama-3-8b-bnb-4bit"
model_name = "unsloth/Llama-3.2-1B-bnb-4bit"

# Define o comprimento máximo da sequência de tokens.
# Tokens são as unidades em que o texto é dividido (palavras, sub-palavras ou caracteres).
# Impacto:
# - Um valor maior permite que o modelo processe contextos mais longos, mas consome mais memória VRAM.
# - Um valor menor economiza memória, mas pode truncar (cortar) exemplos longos, resultando em perda de informação.
# - O valor de 2048 é um bom equilíbrio para a maioria das GPUs de consumidor.
max_seq_length = 2048

# Define o tipo de dado para os cálculos da rede neural.
# Impacto:
# - `None`: Permite que a Unsloth escolha a melhor opção baseada no hardware. Em GPUs modernas, será `bfloat16`.
# - `bfloat16`: (Brain Floating Point) Oferece um bom equilíbrio entre precisão e performance, sendo ideal para treinamento de transformadores.
# - `float16`: Mais propenso a problemas de instabilidade numérica (overflow/underflow), mas rápido.
# - `float32`: Precisão total, mas consome o dobro de memória e é mais lento.
dtype = torch.bfloat16

# Habilita o carregamento do modelo com quantização de 4 bits.
# A quantização reduz a precisão dos pesos do modelo (de 32/16 bits para 4 bits).
# Impacto:
# - Reduz drasticamente o uso de memória VRAM, permitindo que modelos grandes (como Llama 3 8B) rodem em uma única GPU do Colab.
# - Pode causar uma pequena perda de performance, que geralmente é recuperada durante o fine-tuning.
# - É essencial para a viabilidade deste projeto no ambiente proposto.
load_in_4bit = True

# --- Carregamento do Modelo e Tokenizador com Unsloth ---
# A Unsloth aplica patches de otimização para acelerar o carregamento e o treinamento.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b",
    max_seq_length = 2048,
    dtype = torch.float16,   # ou torch.bfloat16
    load_in_4bit = True,
)


==((====))==  Unsloth 2025.9.11: Fast Llama patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [11]:
# --- Configuração do Adaptador LoRA ---
# Adiciona os adaptadores LoRA ao modelo para um fine-tuning eficiente.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","v_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # opcional
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.9.11 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [12]:
def generate_response(instruction: str, input_text: str = "", temperature: float = 0.6) -> str: # LLAMA
    """
    Gera uma resposta a partir do modelo fine-tuned usando um prompt no formato Alpaca,
    com a opção de configurar a temperatura.

    Args:
        instruction (str): A instrução principal para o modelo.
        input_text (str, optional): O texto de entrada que fornece contexto. Defaults to "".
        temperature (float, optional): Controla a aleatoriedade da geração. Valores mais altos
                                        aumentam a criatividade. Defaults to 0.7.

    Returns:
        str: A resposta de texto gerada pelo modelo.
    """
    global model

    # Ensure model is on the correct device (GPU)
    #model.to("cuda")

    # 1. Otimiza o modelo para inferência
    model = FastLanguageModel.for_inference(model)

    # 2. Formata o prompt de entrada
    prompt =  alpaca_prompt_template.format(instruction, input_text, "")

    # 3. Tokeniza o prompt
    tokenized_input = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 4. Gera a sequência de tokens de saída
    generated_tokens = model.generate(**tokenized_input, max_new_tokens=256, temperature=temperature, repetition_penalty=1.1)

    # 5. Decodifica os tokens de volta para texto
    decoded_output = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    # print(decoded_output)
    # 6. Limpa e extrai a resposta final
    final_response = decoded_output.split("### Response:")[-1].split("###")[0].strip()

    return final_response

In [13]:
predicted_content = generate_response(instruction, "Give a description about this book: " + title_question_example)  # Chama a função com o título
print(f"Título/Pergunta: {title_question_example}")
print(f"Conteúdo:\n{predicted_content}") # Imprime o conteúdo

Título/Pergunta: Birds of Britain and Europe with North Africa and the Middle East: Over 3,000 Colour Illustrations (Collins Pocket Guide)
Conteúdo:
The book Birds of Britain and Europe with North Africa and the Middle East: Over 3,000 Colour Illustrations (Collins Pocket Guide) was written by Simon Harrap in 2017. It has 480 pages and it is part of the Collins Pocket Guides series. The book includes over 3,000 color illustrations and covers more than 650 species of birds found in Britain and Europe. It also features sections on bird identification, habitats, migration routes, and conservation issues. This guide is perfect for anyone interested in learning more about birds in their region or who wants to identify different types of birds they see while outdoors.


In [14]:

# --- Configuração do Trainer ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_com_prompt,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        # per_device_train_batch_size: Número de exemplos de treinamento por GPU a cada passo.
        # Impacto: Valores maiores podem acelerar o treino, mas consomem muita VRAM. `1` é seguro para o Colab.
        per_device_train_batch_size = 1,

        # gradient_accumulation_steps: Acumula gradientes por `n` passos antes de atualizar os pesos.
        # Impacto: Simula um batch size maior (1 * 8 = 8) sem aumentar o uso de memória. Essencial para estabilidade.
        gradient_accumulation_steps = 8,

        # warmup_steps: Número de passos iniciais onde a taxa de aprendizado aumenta linearmente.
        # Impacto: Evita que o modelo "divirja" no início do treino.
        warmup_steps = 5,

        # max_steps: Número total de passos de treinamento.
        # Impacto: Controla a duração do fine-tuning. 60 é um valor baixo, para uma demonstração rápida.
        max_steps = 1000,

        #Todo comment
        # num_train_epochs=1,

        # learning_rate: A "velocidade" com que o modelo aprende.
        # Impacto: Valores altos podem causar instabilidade; valores baixos podem tornar o treino muito lento. 2e-4 é um bom padrão para LoRA.
        learning_rate = 10e-4,

        # fp16 / bf16: Habilita o treinamento de precisão mista para economizar memória e acelerar o processo.
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        # logging_steps: Frequência (em passos) para registrar o loss de treinamento.
        logging_steps = 1,

        # optim: Otimizador usado para atualizar os pesos.
        # Impacto: `adamw_8bit` é uma versão do AdamW otimizada para consumir menos memória.
        optim = "adamw_8bit",

        # weight_decay: Termo de regularização para evitar overfitting.
        weight_decay = 0.01,

        # lr_scheduler_type: Estratégia para ajustar a taxa de aprendizado durante o treino (ex: "linear", "cosine").
        lr_scheduler_type = "linear",

        seed = 3407,
        output_dir = "outputs",
        report_to="none",
    ),
)

Map (num_proc=2):   0%|          | 0/75058 [00:00<?, ? examples/s]

In [15]:
dataset_com_prompt[24]

{'text': "Below is an instruction that describes a task, paired with an input that provides further context.\nWrite a response that appropriately completes the request. If you don't know the answer to a question, please don't share false information.\n\n### Instruction:\nGenerate a detailed and informative description for the provided book title.\n\n### Input:\nBirds of Britain and Europe with North Africa and the Middle East: Over 3,000 Colour Illustrations (Collins Pocket Guide)\n\n### Response:\nBrilliantly organised guide ... beautiful, apparently infallible, so comprehensive and above all easy to use. The Sunday Times Superbly handy and comprehensive guide. The Observer<|end_of_text|>"}

In [ ]:
# Faz a limpeza de memória
gc.collect()

# Inicia o treinamento
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 75,058 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 8,388,608 of 6,746,804,224 (0.12% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.147200
2,2.183600
3,2.276200
4,2.086200
5,1.910400
6,1.920000


In [ ]:
# --- Salvando o Modelo Treinado ---

# Salva o modelo LoRA treinado e o tokenizador no diretório especificado.
# O método save_pretrained da Unsloth salva apenas os adaptadores LoRA e a configuração necessária,
# tornando o arquivo menor do que o modelo base completo.
if not os.path.exists(model_directory):
  model.save_pretrained(model_directory)
  tokenizer.save_pretrained(model_directory)
  print(f"Modelo treinado salvo em: {model_directory}")

In [ ]:
# --- Carregando o Modelo Treinado do Google Drive ---



# Define o caminho onde o modelo treinado foi salvo no Google Drive.
# Certifique-se de que este caminho corresponde ao usado na célula anterior (pfx3De0CN1Sa).
adapter_file_path = os.path.join(model_directory, "adapter_model.safetensors")

# Verifica se o diretório do modelo e o arquivo do adaptador existem
if os.path.exists(model_directory) and os.path.isfile(adapter_file_path):
    print(f"Diretório do modelo e arquivo do adaptador encontrados em: {model_directory}")

    # Define o nome do modelo base que foi usado para o fine-tuning.
    # Isso é necessário para carregar a arquitetura e os pesos originais.
    #base_model_name = "unsloth/llama-3-8b-bnb-4bit"
    base_model_name = "unsloth/Phi-4-mini-instruct-bnb-4bit"

    # Carrega o modelo base com as mesmas configurações de quantização e dtype usadas no treinamento.
    # A Unsloth permite carregar o modelo base e, em seguida, anexar os adaptadores LoRA.
    # load_in_4bit deve ser False aqui para carregar o modelo base em sua precisão original antes de anexar os adaptadores.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = base_model_name,
        max_seq_length = 2048, # Usar o mesmo max_seq_length do treinamento
        dtype = None, # Usar o mesmo dtype (None para auto) do treinamento
        load_in_4bit = False, # Carregar o modelo base sem quantização de 4 bits inicialmente
    )

    # Carrega os adaptadores LoRA salvos do diretório especificado.
    # Isso adiciona os pesos treinados ao modelo base.
    # Certifique-se de que o modelo está na GPU antes de carregar o estado, se necessário.
    model.to("cuda") # Move o modelo para a GPU
    FastLanguageModel.set_peft_model_state_dict(model, torch.load(adapter_file_path, map_location="cuda")) # Carrega para a GPU

    print(f"Modelo treinado carregado com sucesso de: {model_directory}")

else:
    print(f"Erro: Diretório do modelo ou arquivo do adaptador não encontrado em: {model_directory}")
    print("Por favor, verifique se o treinamento foi concluído com sucesso e se o modelo foi salvo corretamente.")
    model = None # Define model como None para indicar que o carregamento falhou
    tokenizer = None # Define tokenizer como None

In [ ]:
predicted_content = generate_response(instruction, "Who wrote this book: " + title_question_example)  # Chama a função com o título
print(f"Título/Pergunta: {title_question_example}")
print(f"Conteúdo:\n{predicted_content}") # Imprime o conteúdo

In [ ]:
# TODO
# Implementar RAG com os dados para manipular os prompts e também adicionar 5 casos de testes no LLM as a Judge
!pip install google.generativeai
!pip install chromadb
!pip install google.api_core


In [ ]:
import google.generativeai as genai
import chromadb
from google.api_core import retry
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.colab import userdata


api_key = userdata.get('GEMINI_PRO_AI_API_KEY')
genai.configure(api_key=api_key)

In [ ]:


# Faz a limpeza de memória
gc.collect()


def load_documents_from_dataset(dataset):
    """
    Lê o arquivo output.json e retorna uma lista de strings no formato:
    Term: <name>. Description: <description>
    """
    documents = []
    i = 0
    for item in dataset:
        i=i + 1
        title = item.get('title', '').strip()
        content = item.get('content', '').strip().split('.')[0]
        if title and content:
            documents.append(f"Title:{title}.Content:{content}")
        if i == 1000:
            break
    return documents

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self, document_mode=True):
        self.document_mode = document_mode

    def __call__(self, input: Documents) -> Embeddings:
        task_type = "retrieval_document" if self.document_mode else "retrieval_query"
        response = genai.embed_content(
            model="models/text-embedding-004",
            content=input,
            task_type=task_type,
            request_options={"retry": retry.Retry(
                predicate=retry.if_transient_error)}
        )
        return response["embedding"]

# Database operations

class DocumentDatabase:
    def __init__(self, db_name="titlesAmazon", persist_directory="./chromadb_storage"):
        self.db_name = db_name
        self.client = chromadb.PersistentClient(path=persist_directory)

    def get_db(self, document_mode=True):
        return self.client.get_or_create_collection(
            name=self.db_name,
            embedding_function=GeminiEmbeddingFunction(document_mode)
        )
    def clean_db(self, document_mode=True):
        return self.client.delete_collection(name=self.db_name)

    def store_documents(self, documents):
        db = self.get_db(True)
        if db.count() == 0:
            db.add(documents=documents, ids=[str(i) for i in range(len(documents))])
        return db.count()

    def query(self, question):
        db = self.get_db(False)
        result = db.query(query_texts=[question], n_results=1)
        return result["documents"][0][0]

def get_answer(question, db):
    # Get model and reference data
    flash = genai.GenerativeModel('gemini-1.5-flash')
    reference = db.query(question)

    # Clean text and create prompt
    prompt = f"""

    Below is an instruction that describes a task, paired with an input that provides further context.
    Write a response that appropriately completes the request.

    ### Instruction:
    Give the description of this item.

    ### Input:
    {question}

    ### Reference that you can use to give the information:
    {reference.replace("\n", " ")}
"""

    # return flash(prompt)
    response = flash.generate_content(prompt)
    return response.text

DOCUMENTS = None
DOCUMENTS = load_documents_from_dataset(dataset_formatado)
db = DocumentDatabase()
db.clean_db()
for i in range(0, len(DOCUMENTS), 10):  # batch de 10
    batch = DOCUMENTS[i:i+10]
    db.store_documents(batch)

# doc_count = db.store_documents(DOCUMENTS)
print(f"Stored {doc_count} documents in the database")

In [ ]:
title_question_example = "Autumn Story Brambly Hedge"  # Exemplo de título como pergunta
reference = db.query( question = title_question_example)
print(reference)
# answer = get_answer(title_question_example, db)

In [ ]:
# LLM as a Judge - Testes de LLM utilizando a API do Gemini

os.environ["CONFIDENT_AI_API_KEY"] = userdata.get('CONFIDENT_AI_API_KEY')
GOOGLEAI_API_KEY = userdata.get('GEMINI_PRO_AI_API_KEY')

from deepeval import assert_test
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase


from deepeval.models import GeminiModel

EVAL_MODEL = "gemini-2.0-flash"


eval_model = GeminiModel(
    model_name=EVAL_MODEL,
    api_key=GOOGLEAI_API_KEY,
    temperature=0
)



# 1. Preparar os dados para o teste
# Usaremos o mesmo exemplo dos testes manuais para consistência.
test_title = "A Lifetime of Secrets: A PostSecret Book"
ground_truth_content = "The award-winning PostSecret project's most profound and stunning postcards to dateFor the past three years Frank Warren has invited people of all backgrounds and nationalities to send him creatively decorated postcards bearing secrets they have never before revealed. He has shared these PostSecrets on his award-winning blog, www.PostSecret.com, in an internationally traveling art exhibit, and in three electrifying books: the bestsellingPostSecret, My Secret,andThe Secret Lives of Men and Women.Now, in his most extraordinary book yet, Warren again delves into our collective confessions, presenting a never-before-seen selection of provocative and moving PostSecrets.A Lifetime of Secretslays bare our private fears, hopes, regrets, and desires, from people as young as eight and as old as eighty. From painful admissions of infidelity to breathtaking revelations and endearing sentiments, Warren's latest collection will shock and move readers of every age, revealing secrets that have haunted their creators for a lifetime.Six PostSecrets fromA Lifetime of SecretsHere are six of the PostSecrets included inA Lifetime of Secrets, and never before seen online. Click on each image to see a larger version.Frank Warren's Introduction toA Lifetime of SecretsWhen I told my father I was collecting secrets from strangers for an art project, he didn&#x92;t know what to think. I tried to explain how the thousands of secrets that had been mailed to me were more than mere confessions. They could be beautiful, funny, sorrowful, inspiring.\"But, Frank,\" he asked, \"why are you soliciting secrets from strangers, and why would anyone tell you a real secret?\"I invited my father to fly out for a PostSecret art exhibit in Washington, D.C., where hundreds of the postcards were on display. More than 15,000 people came to see the secrets, and my father was there, day after day, to hear many of their transformative stories. Some people told me they recognized a hidden part of themselves on a stranger&#x92;s postcard. Others shared personal experiences of how talking about a painful secret had helped heal a lifelong relationship.The exhibit came to an end and I took my father back to the airport to catch a red-eye flight home. During our drive we passed through a long dark stretch of highway when my father broke the silence by asking me, \"Do you want to know my secret?\" He bravely recounted a traumatic childhood experience. When he finished, we had a true talk that gave me a richer understanding of my father and recast our relationship.&#x95; &#x95; &#x95;ForA Lifetime of Secrets, the fourth PostSecret book, I've selected postcards that show how secrets can reveal a momentary impulse or haunt us for decades and arranged them by age to follow the common journey we all take through childhood, adolescence, adulthood, maturity. Stretched over a full lifespan, the secrets expose the meaningful ways we change over time, and the surprising ways we don't.The postcards narrate childhood stories that have never been spoken; they voice the guarded confessions of our parents and grandparents. They confirm that our rich interior lives are not defined by how old we are, and that with aging comes not only loss but also the possibility of grace and wisdom.The following two secrets arrived in my mailbox the same week. The postmarks on each card were different, but when I posted them together on the PostSecret website (www.postsecret.com) they seemed as though they could have been written by the same person at two different points in her life.I am a junior in high school. I have good friends and a loving family. I am smart. I am a good athlete and musician. But I would trade all that in if it meant I would be beautiful.I spent my high school years believing I was UGLY. I just went through a photo album that had pictures of me over the last 20 years. Turns out I was/am kind of cute. No more wasting time on thinking otherwise.&#x95; &#x95; &#x95;When I give PostSecret presentations at college campuses, my hope is that people I have never met will be inspired to change their lives through the secrets and stories being shared. Not long ago, at one of my talks, it was my life that was changed, and the secret that inspired me came from a stranger in the front row.I began my presentation by handing out blank postcards to everyone in the auditorium. I invited each person to anonymously write down a secret on a card and then pass it on. For the next hour, the postcards circulated and were read silently multiple times. At the end of my talk, I asked if anyone would like to stand and read the secret they were holding at that moment. A man in the front row stood up and haltingly read:I wish I could apologize to my younger brother for the way I treated him growing up.He sat down and exchanged a long look with the young man next to him. After more volunteers read aloud some of the other secrets that had been passed around, I collected all the cards. The man in the front row handed me the postcard he had read from, and the two men walked out together.His postcard was blank.I have witnessed many times how the courage of sharing a secret can be contagious. When I realized that the man had been pretending to read someone else&#x92;s secret and that the person he had left with was likely his brother, I was inspired.Growing up, I was not an ideal older brother. As an adult, I have wished for an opportunity to apologize for some of my actions but did not want to open old wounds. I have not shared this secret with my brother . . . until now.--Frank Warren"

# 2. Gerar a resposta do modelo fine-tuned
# A função `test_model_alpaca` já foi definida anteriormente
model_generated_content = generate_response(
    instruction="DESCRIBE THE TITLE BASED ON THE CONTENT",
    input_text=test_title
)

print(f"Input (Título): {test_title}")
print(f"Actual Output (Gerado): {model_generated_content}")
print(f"Expected Output (Referência): {ground_truth_content}")

# 3. Criar o Caso de Teste
test_case = LLMTestCase(
    input=test_title,
    actual_output=model_generated_content,
    expected_output=ground_truth_content
)

# 4. Definir a Métrica
# Usaremos a AnswerRelevancyMetric, que avalia a relevância da resposta em relação à pergunta.
# O threshold define a pontuação mínima para o teste passar (ex: 0.5).
relevancy_metric = AnswerRelevancyMetric(
    model=eval_model,
    threshold=0.6
)

# 5. Executar a Avaliação
# `assert_test` executa a métrica no caso de teste e lança um erro se falhar.
# Para rodar múltiplos testes, use `deepeval.evaluate([test_case], [relevancy_metric])`
assert_test(test_case, [relevancy_metric])

print("\n✅ Teste de Relevância da Resposta concluído com sucesso!")